# 1-board: terminal-chain

Measure the terminal taps, the rail tap, the bias node and the ADC reference against the meter, so a count on any voltage channel is a graded number of volts.

## Purpose

The firmware reads the motor terminals through two dividers returned to a bias node, the supply rail through a third, and every count is a share of the 3.3 V reference. Back-EMF, winding resistance, the undervoltage trip and the rail-fold all come through these. This experiment grades each against the meter on DC laps, where the terminals sit at steady voltages, and reads the reference and the bias node directly.

## Procedure

The DC laps of current-scale, with the DMM on DC V at one node per pass. The bare load is on the terminals throughout.

1. `capture.py load`: the meter across the load at the motor terminals, MOT_A to MOT_B, which the difference of the two taps should read.
2. `capture.py tap_a`: MOT_A to GND, the absolute voltage one tap reads against the bias node.
3. `capture.py rail`: VSYS to GND, the rail behind the feed diode, at rest and sagging under the lap.
4. `capture.py vdd`: the 3V3 test point to GND, at rest only, five seconds at each level.

## Data

- `<node>/levels.csv`, `<node>/laps/<k>/`, `<node>/dmm.csv` as in current-scale.
- `vdd/rest-<V>V/` and `vdd/dmm.csv`: rests only.

### The passes

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from oscnb import b41t, boards, display, laps, results, tel
from oscnb.display import show

BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#898781"
BOARD = boards.BOARDS[boards.DEFAULT]
CAPTURE = Path("../../../telemetry") / BOARD.key / "terminal-chain" / "capture-1"
CAPTURE = CAPTURE.resolve()
print(BOARD.key, "from", CAPTURE)

In [ ]:
nodes = [p.name for p in CAPTURE.iterdir() if (p / "dmm.csv").exists()]
print("passes:", nodes)
dmm = {n: b41t.load(CAPTURE / n / "dmm.csv") for n in nodes}
fig, ax = plt.subplots(1, len(nodes), figsize=(5 * len(nodes), 3.5), squeeze=False)
for a, n in zip(ax[0], nodes):
    d = dmm[n]; a.plot(d["t"], d["value"], ".", ms=2, color=BLUE); a.set_title(f"{n}: the meter (V)"); a.set_xlabel("time (s)"); a.grid(alpha=0.3)
plt.show()

The meter's log of each pass. The lap passes step through the levels with a rest between; the vdd pass sits at one value.

## Method

Each lap gives one point per node: the chip's reading converted at the registry's nominal ratio against the meter's volts in the same window. The ratio of the two, pooled over the laps, is the chain's scale error for that node. The taps' difference is bias free; the single tap needs the bias node, taken from the rest of the same recording, and its error says whether the bias network and the ratio are right together. The rail's reading at rest and under the lap against the meter gives its scale and, with the lap current from the current-scale result, the supply's source impedance. The reference is the meter's reading in the vdd pass.

### The lap table per node

In [ ]:
lap_nodes = [n for n in nodes if n != "vdd"]
T = {}
for n in lap_nodes:
    t = laps.table(CAPTURE / n, BOARD, dmm[n])
    t["diff_v"] = BOARD.diff_v(t["vmotor_a"], t["vmotor_b"])
    t["tap_a_v"] = BOARD.term_v(t["vmotor_a"], t["vb_rest"])
    t["rail_v"] = BOARD.rail_v(t["vbus_raw"]); t["rail_rest_v"] = BOARD.rail_v(t["rail_rest"])
    T[n] = t
    chip = {"load": t["diff_v"].abs(), "tap_a": t["tap_a_v"], "rail": t["rail_v"]}[n]
    t["chip"] = chip
    t["error_pct"] = (chip / t["meter"].abs() - 1) * 100
    show(t[["k", "volts_set", "dir", "vb_rest", "chip", "meter", "error_pct", "meter_n"]], n=None)

One table per node: the chip's reading at the nominal conversion, the meter, and the chip's error as a share of the meter. For tap_a only the laps that drive A high read a terminal voltage worth grading; the low-side laps read the return path's drop.

In [ ]:
out = {}
if "load" in T:
    e = T["load"]["error_pct"].dropna(); out["terminal_scale_pct"] = (e.mean(), e.min(), e.max())
    print(f"terminal difference: chip reads {e.mean():+.2f} % of the meter over {len(e)} laps ({e.min():+.2f} to {e.max():+.2f})")
if "tap_a" in T:
    t = T["tap_a"]; hi = t[t["vmotor_a"] > t["vmotor_b"]]
    e = hi["error_pct"].dropna(); out["tap_abs_pct"] = (e.mean(), e.min(), e.max())
    lo = t[t["vmotor_a"] <= t["vmotor_b"]]
    print(f"tap A on the high side: {e.mean():+.2f} % ({len(e)} laps); on the low side the tap reads {lo['chip'].mean():.3f} V against {lo['meter'].abs().mean():.3f} V on the meter")
if "rail" in T:
    t = T["rail"]; e = t["error_pct"].dropna(); out["rail_scale_pct"] = (e.mean(), e.min(), e.max())
    sag = (t["rail_rest_v"] - t["rail_v"])
    cs = results.read("1-board", "current-scale", pair=BOARD.key) if (Path("../current-scale/results") / f"{BOARD.key}.json").exists() else None
    if cs:
        amps = (t["current_raw"] - t["zero"]) / cs["counts_per_a"]["value"]
        z = (sag / amps).replace([np.inf, -np.inf], np.nan).dropna()
        out["source_impedance_ohm"] = (z.mean(), z.min(), z.max())
        print(f"rail: {e.mean():+.2f} % of the meter; sags {sag.mean():.3f} V under the laps, {z.mean():.3f} ohm of source impedance")
    else:
        print(f"rail: {e.mean():+.2f} % of the meter; sags {sag.mean():.3f} V under the laps (no current-scale result yet for the impedance)")
vb = pd.concat([t["vb_rest"] for t in T.values()])
print(f"bias node at rest {vb.mean():.1f} counts ({vb.min():.0f} to {vb.max():.0f}) against {BOARD.bias_nom_counts} nominal")

The scale error per node pooled over the laps, the rail's sag, and the bias node at rest against its nominal.

### The reference

In [ ]:
vdd = None
if "vdd" in nodes:
    d = dmm["vdd"]["value"].dropna(); vdd = (d.median(), d.quantile(0.05), d.quantile(0.95))
    print(f"VDD {vdd[0]:.4f} V on the meter ({vdd[1]:.4f} to {vdd[2]:.4f}) against {BOARD.vdd_mv} mV in the table: every count is {(vdd[0] * 1e3 / BOARD.vdd_mv - 1) * 100:+.2f} % off by the reference alone")
    rests = sorted(CAPTURE.glob("vdd/rest-*"))
    rr = []
    for p in rests:
        df, meta = tel.load(p, BOARD); b = tel.baseline(df)
        rr.append({"rest": p.name, "vb_a": b["vmotor_a"].mean(), "vb_b": b["vmotor_b"].mean(), "rail_counts": b["vbus_raw"].mean()})
    show(pd.DataFrame(rr), n=None)
else:
    print("no vdd pass")

The reference on the meter, and the rests the vdd pass recorded at each level, which show whether the bias node and the rail tap move with the supply at rest.

## Results

The scale error per node with its range over the laps as the bounds, the bias node at rest, the reference, and the source impedance when the current-scale result exists.

In [ ]:
q = {k: results.quantity(v[0], lo=v[1], hi=v[2], unit="%" if k.endswith("pct") else "ohm") for k, v in out.items()}
results.write("1-board", "terminal-chain", CAPTURE, pair=BOARD.key,
              notes="DC laps at 100 % duty on the bare load, DMM on DC V at one node per pass; chip at the registry's nominal ratios against the meter",
              vb_rest_counts=results.quantity(vb.mean(), lo=vb.min(), hi=vb.max(), unit="counts"),
              **({"vdd_v": results.quantity(vdd[0], lo=vdd[1], hi=vdd[2], unit="V")} if vdd else {}), **q)

## Limitations

- The meter's DC V accuracy bounds the scale errors; the dmm experiment's result is to carry that number once the manual's figure is added.
- The taps are graded at DC. Under switching the tap samples land on edges, which is the sample-timing experiment.
- The low side of a driven terminal, the switch and the shunt and the copper, is in tap A's low-side laps and not separated here.

## Open questions

- Whether the bias node moves under drive. It rises with the leg current through the taps' bottoms, and a meter at the bias node during a lap would show by how much against what the registry's network predicts.

## Conclusion

To be written from the capture.